# Feature importance analysis
In this demo, we will implement Recursive Feature Elimination (RFE) with Random Forest to select the optimal subset of features for DDoS attack detection. The goal is to reduce training and inference time while ensuring the desired level of classification accuracy. 
We will use a dataset of benign and various DDoS attacks from the CIC-DDoS2019 dataset (https://www.unb.ca/cic/datasets/ddos-2019.html).
The network traffic has been previously pre-processed in a way that packets are grouped in bi-directional traffic flows using the 5-tuple (source IP, destination IP, source Port, destination Port, protocol). Each flow is represented with 21 packet-header features computed from max 1000 packets:

| Feature nr.         | Feature Name |
|---------------------|---------------------|
| 00 | timestamp (mean IAT) | 
| 01 | packet_length (mean)| 
| 02 | IP_flags_df (sum) |
| 03 | IP_flags_mf (sum) |
| 04 | IP_flags_rb (sum) | 
| 05 | IP_frag_off (sum) |
| 06 | protocols (mean) |
| 07 | TCP_length (mean) |
| 08 | TCP_flags_ack (sum) |
| 09 | TCP_flags_cwr (sum) |
| 10 | TCP_flags_ece (sum) |
| 11 | TCP_flags_fin (sum) |
| 12 | TCP_flags_push (sum) |
| 13 | TCP_flags_res (sum) |
| 14 | TCP_flags_reset (sum) |
| 15 | TCP_flags_syn (sum) |
| 16 | TCP_flags_urg (sum) |
| 17 | TCP_window_size (mean) |
| 18 | UDP_length (mean) |
| 19 | ICMP_type (mean) |
| 20 | Packets (counter)|

In [ ]:
# Author: Roberto Doriguzzi-Corin
# Project: Course on Network Intrusion and Anomaly Detection with Machine Learning
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#   http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import os
import numpy as np
import pandas as pd
import glob
import h5py
import time
import sys
import copy
import argparse
from sklearn.metrics import classification_report, f1_score, accuracy_score
import logging
from util_functions import *
from IPython.display import Image, display
from sklearn.tree import export_graphviz
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

SEED=1
np.random.seed(SEED)

OUTPUT_FILE = "./rf_tree"
DATASET_FOLDER = "./DOS2019"

from matplotlib import pyplot as plt
plt.rcParams.update({'figure.figsize': (12.0, 8.0)})
plt.rcParams.update({'font.size': 14})

feature_names = get_feature_names()
target_names = ['benign', 'dns',  'syn', 'udplag', 'webddos']
X_train, y_train = load_dataset(DATASET_FOLDER + "/*" + '-train.hdf5')
X_test, y_test = load_dataset(DATASET_FOLDER + "/*" + '-test.hdf5')

# Feature selection with Recursive Feature Elimination (RFE)
Decide the maximum number of features to keep and the minimum accuracy that you allow. We do not use the function from the scikit-learn library because we also check the accuracy

In [ ]:
def RFE(X_train, y_train, feature_names, n_features_to_select):
    """
    Custom implementation of Recursive Feature Elimination (RFE) with Random Forest Classifier.
    
    Parameters:
    X_train (ndarray): Training data.
    y_train (ndarray): Training labels.
    feature_names (ndarray): Feature names.
    n_features_to_select (int): Number of features to select.
    
    Returns:
    selected_features (list): List of the selected feature names.
    feature_ranking (list): List of the ranking of each feature at the end.
    """
    # Keep track of the current feature set
    current_features = np.arange(X_train.shape[1])
    
    # Initialize feature rankings (all features start with rank 1)
    feature_ranking = np.ones(X_train.shape[1], dtype=int)
    
    # Iteratively eliminate features until the desired number is reached
    while len(current_features) > n_features_to_select:
        # Train a RandomForestClassifier
        rf = RandomForestClassifier(n_estimators=100, random_state=SEED)
        rf.fit(X_train[:, current_features], y_train)
        
        # Get the feature importances
        feature_importances = rf.feature_importances_
        
        # Find the index of the least important feature
        least_important_idx = np.argmin(feature_importances)
        
        # Update the ranking for the removed feature
        feature_ranking[current_features[least_important_idx]] = len(current_features)
        
        # Remove the least important feature from the current set
        current_features = np.delete(current_features, least_important_idx)
    
    # The remaining features are the selected ones
    selected_features = current_features
    fn = np.array(feature_names)
    return fn[selected_features], selected_features, feature_ranking

In [ ]:
# Define the number of features we want to select
n_features_to_select = 5

# Run the custom RFE algorithm
selected_features, selected_feature_indeces, feature_ranking = RFE(X_train, y_train, feature_names, n_features_to_select)

# Display the selected features and the final rankings
print("Selected Features:", selected_features)
print("Final Feature Ranking (1 = selected, higher values indicate less important features):")
for feature, rank in zip(feature_names, feature_ranking):
    print(f"{feature}: {rank}")

# Plot the final feature ranking
ranking_df = pd.DataFrame({'Feature': feature_names, 'Ranking': feature_ranking})
ranking_df.sort_values(by='Ranking', inplace=True)

plt.barh(ranking_df['Feature'], ranking_df['Ranking'])
plt.xlabel('Ranking')
plt.ylabel('Features')
plt.title('Feature Ranking by Custom RFE with Random Forest Classifier')
plt.show()

# Train and test an RF model with the selected features
We now compare an RF model trained with all the features against another one trained with only the most important features.

In [ ]:
rf_full = RandomForestClassifier(n_estimators=100,max_depth=3,min_samples_split=50,oob_score=True, random_state=SEED)
rf_full.fit(X_train, y_train)

rf_optimized = RandomForestClassifier(n_estimators=100,max_depth=3,min_samples_split=50,oob_score=True, random_state=SEED)
rf_optimized.fit(X_train[:, selected_feature_indeces], y_train)

print("Full feature set:")
y_pred = rf_full.predict(X_test)
print(classification_report(y_test, y_pred, target_names=target_names))

print("Optimized feature set:")
y_pred = rf_optimized.predict(X_test[:, selected_feature_indeces])
print(classification_report(y_test, y_pred, target_names=target_names))


# Importance with feature reset
Use the above code to implement a feature ranking algorithm by resetting one feature at a time.
In this case, you might obtain negative values for some features. This means that those features can be noisy, irrelevant or redundant. 
This method can be used with any ML algorithm. In this case, we try with a Multi-Layer Perceptron.

| <img src="./mlp-feature-importance.png" width="90%">  |
|--|
| MLP model|

In the first step, we import the necessary libraries and define the model.

In [ ]:
# Model definition
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential, load_model,save_model
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.utils import plot_model
from tensorflow.keras.optimizers import Adam
# disable GPUs for test reproducibility
tf.config.set_visible_devices([], 'GPU')
tf.keras.utils.set_random_seed(SEED)

def nn_model(input_features = 21):
    # Define a simple neural network model
    mlp_model = Sequential(name="MLP", layers=[
        Dense(64, input_shape=(input_features,), activation='relu'),
        Dense(32, activation='relu'),
        Dense(5, activation='softmax')
    ])

    # Compile the model
    mlp_model.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])
    # Train the model and pass the TensorBoard callback
    print (mlp_model.summary())

    # Visualize the model architecture
    plot_model(mlp_model, to_file='mlp_binary_plot.png', show_shapes=True, show_layer_names=True)
    return mlp_model

# Model training
We train the MLP model with the original features in order to determine a baseline accuracy.


In [ ]:
X_train, y_train = load_dataset(DATASET_FOLDER + "/*" + '-train.hdf5')
y_train = np.eye(5)[y_train] # from integer to one-hot 
X_val, y_val = load_dataset(DATASET_FOLDER + "/*" + '-val.hdf5')
y_val = np.eye(5)[y_val] # from integer to one-hot 
X_test, y_test = load_dataset(DATASET_FOLDER + "/*" + '-test.hdf5')
y_test = np.eye(5)[y_test] # from integer to one-hot 

mlp_model = nn_model(X_train.shape[1])

history = mlp_model.fit(X_train, y_train, epochs=100, validation_data=(X_val, y_val), batch_size=16, verbose=1)
### Predict with the trained model
predictions = mlp_model.predict(X_test)
# from probabilities to one-hot-encoding
max_indices = np.argmax(predictions, axis=1)
y_pred = np.zeros_like(predictions)
y_pred[np.arange(predictions.shape[0]), max_indices] = 1

### Here we save the accuracy score that we use as a baseline for evaluating the importance of each feature
baseline_accuracy = accuracy_score(y_test,y_pred)
print ("Baseline test accuracy: ", baseline_accuracy)

# Feature importance
Now, we measure the importance of the features by setting them to zero, one after the other. By measuring the decrease in accuracy, we establish the importance of each feature. 

In [ ]:
feature_ranking = {}
for feature_index in range(len(feature_names)):
    X_test_redux = copy.deepcopy(X_test)
    ### Set one to zero one feature at a time using the feature_index and the np.zeros method
    X_test_redux[:,feature_index] = 0

    #history = mlp_model.fit(X_train_redux, y_train, epochs=100, validation_data=(X_val_redux, y_val), batch_size=16, verbose=0)

    ### Predict using the new test set with one feature set to zero
    predictions = mlp_model.predict(X_test_redux)
    max_indices = np.argmax(predictions, axis=1)
    y_pred_redux = np.zeros_like(predictions)
    y_pred_redux[np.arange(predictions.shape[0]), max_indices] = 1

    # Here we save the difference in accuracy with the baseline
    feature_ranking[feature_names[feature_index]] = baseline_accuracy - accuracy_score(y_test,y_pred_redux)

print (feature_ranking)
plt.barh(feature_names, feature_ranking.values())
plt.show()

# Feature importance with permutations
Similar to feature reset, this method measures the decrease in model performance (e.g., accuracy, RMSE) when the values of a specific feature are randomly shuffled. If shuffling a feature decreases performance significantly, that feature is deemed important. 

In [ ]:
feature_ranking = {}
for feature_index in range(len(feature_names)):
    X_test_redux = copy.deepcopy(X_test)
    ### Shuffling the values of a feature
    np.random.shuffle(X_test_redux[:,feature_index])

    ### Predict using the new test set with one feature set to zero
    predictions = mlp_model.predict(X_test_redux)
    max_indices = np.argmax(predictions, axis=1)
    y_pred_redux = np.zeros_like(predictions)
    y_pred_redux[np.arange(predictions.shape[0]), max_indices] = 1

    # Here we save the difference in accuracy with the baseline
    feature_ranking[feature_names[feature_index]] = baseline_accuracy - accuracy_score(y_test,y_pred_redux)

print (feature_ranking)
plt.barh(feature_names, feature_ranking.values())
plt.show()

# Feature importance with Partial Dependence Plots (PDP)

In [ ]:
def compute_pdp(model, X, feature_index, grid_resolution=100):
    # Create grid of values for the feature
    feature_values = np.linspace(X[:, feature_index].min(), X[:, feature_index].max(), grid_resolution)
    
    pdp = []
    
    # Iterate through the values of the feature
    for value in feature_values:
        X_copy = X.copy()
        X_copy[:, feature_index] = value  # Set the feature column to a fixed value
        
        # Predict using the Keras model
        predictions = model.predict(X_copy)
        
        # Calculate the average prediction for each value
        pdp.append(np.mean(predictions, axis=0))
    
    return feature_values, np.array(pdp)

In [ ]:
X_train, y_train = load_dataset(DATASET_FOLDER + "/*" + '-train.hdf5')
y_train = np.eye(5)[y_train] # from integer to one-hot 
X_val, y_val = load_dataset(DATASET_FOLDER + "/*" + '-val.hdf5')
y_val = np.eye(5)[y_val] # from integer to one-hot 
X_test, y_test = load_dataset(DATASET_FOLDER + "/*" + '-test.hdf5')
y_test = np.eye(5)[y_test] # from integer to one-hot 

mlp_model = nn_model()
history = mlp_model.fit(X_train, y_train, epochs=100, validation_data=(X_val, y_val), batch_size=16, verbose=0)

# Compute PDP for each feature

line_styles = ['-', '--', '-.', ':', (0, (5, 10))]  # Solid, dashed, dash-dot, dotted, loosely dashed

for feature_index in range(X_train.shape[1]):
    feature_values, pdp = compute_pdp(mlp_model, X_train, feature_index=feature_index)
    print (feature_names[feature_index])

    # Plot the PDP
    #plt.plot(feature_values, pdp)
    for class_idx in range(len(target_names)):  
        plt.plot(feature_values, pdp[:, class_idx], label=target_names[class_idx], 
        linestyle=line_styles[class_idx % len(line_styles)])

    plt.xlabel("Values of feature: " + feature_names[feature_index])
    plt.ylabel("Partial Dependence (probability of each class)")
    plt.title("Partial Dependence Plot for Feature: " + feature_names[feature_index])
    plt.legend(loc='best') 
    plt.show()